### Librerie

In [1]:
import pandas as pd
import requests
from datetime import datetime
from pathlib import Path
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import cross_val_score,train_test_split,StratifiedKFold,GridSearchCV 
from sklearn.metrics import accuracy_score,recall_score,classification_report,confusion_matrix
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [2]:
import pandas as pd
import numpy as np

class SerieADataProcessor:
    def __init__(self, campionati_dict):
        self.raw_data = campionati_dict
        self.final_dataset = pd.DataFrame()
        self.features = []
        self.model = None # Verrà impostato esternamente[cite: 1]

    def _parse_date_safe(self, date_str):
        for fmt in ("%d/%m/%Y", "%d/%m/%y", "%Y-%m-%d", "%d-%m-%Y"):
            try: return pd.to_datetime(str(date_str), format=fmt)
            except: continue
        return pd.NaT

    def _calculate_winter_stats(self, df_season, season_id):
        # Selezione colonne core dal notebook originale[cite: 1]
        m = df_season[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]].copy()
        m["Date"] = m["Date"].apply(self._parse_date_safe)
        m = m.sort_values("Date").dropna(subset=["FTHG", "FTAG"]).reset_index(drop=True)

        teams = pd.unique(pd.concat([m["HomeTeam"], m["AwayTeam"]]))
        games_in_half_season = len(teams) - 1
        
        stats = pd.DataFrame(index=teams, data={
            "GP": 0, "W": 0, "D": 0, "L": 0, "GF": 0, "GA": 0, "Pts": 0
        })

        winter_record = None
        for _, row in m.iterrows():
            h, a, hg, ag = row["HomeTeam"], row["AwayTeam"], int(row["FTHG"]), int(row["FTAG"])
            for team, g_for, g_against in [(h, hg, ag), (a, ag, hg)]:
                stats.loc[team, "GP"] += 1
                stats.loc[team, "GF"] += g_for
                stats.loc[team, "GA"] += g_against
                if g_for > g_against: 
                    stats.loc[team, "Pts"] += 3; stats.loc[team, "W"] += 1
                elif g_for < g_against: 
                    stats.loc[team, "L"] += 1
                else: 
                    stats.loc[team, "Pts"] += 1; stats.loc[team, "D"] += 1

            if winter_record is None and stats["GP"].min() >= games_in_half_season:
                table = stats.copy()
                table["GD"] = table["GF"] - table["GA"]
                table = table.sort_values(by=["Pts", "GD", "GF"], ascending=False)
                champ = table.index[0]
                winter_record = table.iloc[0].to_dict()
                winter_record.update({
                    "Team": champ, "Season": season_id,
                    "GapOnSecond": winter_record["Pts"] - table.iloc[1]["Pts"],
                    "WinRate": winter_record["W"] / winter_record["GP"],
                    "AvgGoalScored": winter_record["GF"] / winter_record["GP"],
                    "AvgGoalConceded": winter_record["GA"] / winter_record["GP"]
                })

        final_winner = stats.assign(GD=stats.GF-stats.GA).sort_values(["Pts","GD","GF"], ascending=False).index[0]
        if winter_record:
            winter_record["WonScudetto"] = 1 if winter_record["Team"] == final_winner else 0
            return winter_record
        return None

    def build_training_set(self):
        records = []
        for sid, df in self.raw_data.items():
            try:
                record = self._calculate_winter_stats(df, sid)
                if record: records.append(record)
            except Exception as e:
                print(f"Errore stagione {sid}: {e}")
        
        df = pd.DataFrame(records)
        cols_id = ["Team", "Season"]
        col_target = ["WonScudetto"]
        self.features = [c for c in df.columns if c not in cols_id + col_target]
        self.final_dataset = df[cols_id + self.features + col_target]
        return self.final_dataset

    def set_trained_model(self, model):
        """Metodo SWE: Dependency Injection del modello addestrato esternamente."""
        self.model = model

    def predici_scudetto(self, nome_squadra, dati_attuali):
        """Esegue la predizione usando il modello iniettato."""
        if self.model is None:
            raise ValueError("Il modello non è stato impostato. Usa processor.set_trained_model(model).")
            
        input_df = pd.DataFrame([dati_attuali])
        input_df = input_df[self.features] # Garantisce l'allineamento delle feature[cite: 1]
        
        prob = self.model.predict_proba(input_df)[0]
        
        print(f"\n--- Analisi Modello: {nome_squadra} ---")
        print(f"Probabilità vittoria finale: {prob[1]:.2%}")
        return prob[1]

In [3]:
def season_folder(start_year: int) -> str:
    # 1999 -> "9900", 2000 -> "0001", 2023 -> "2324"
    return f"{start_year % 100:02d}{(start_year + 1) % 100:02d}"

campionati = {}

for start_year in range(1993, 2025):  
    stagione = season_folder(start_year)
    url = f"https://www.football-data.co.uk/mmz4281/{stagione}/I1.csv"
    try:
        df = pd.read_csv(url)
        campionati[stagione] = df
        print("Scaricata:", stagione, "->", start_year, "/", start_year + 1)
    except Exception as e:
        print("Non trovata:", stagione, "-", e.__class__.__name__)

Scaricata: 9394 -> 1993 / 1994
Scaricata: 9495 -> 1994 / 1995
Scaricata: 9596 -> 1995 / 1996
Scaricata: 9697 -> 1996 / 1997
Scaricata: 9798 -> 1997 / 1998
Scaricata: 9899 -> 1998 / 1999
Scaricata: 9900 -> 1999 / 2000
Scaricata: 0001 -> 2000 / 2001
Scaricata: 0102 -> 2001 / 2002
Scaricata: 0203 -> 2002 / 2003
Non trovata: 0304 - ParserError
Non trovata: 0405 - ParserError
Scaricata: 0506 -> 2005 / 2006
Scaricata: 0607 -> 2006 / 2007
Scaricata: 0708 -> 2007 / 2008
Scaricata: 0809 -> 2008 / 2009
Scaricata: 0910 -> 2009 / 2010
Scaricata: 1011 -> 2010 / 2011
Scaricata: 1112 -> 2011 / 2012
Scaricata: 1213 -> 2012 / 2013
Scaricata: 1314 -> 2013 / 2014
Scaricata: 1415 -> 2014 / 2015
Scaricata: 1516 -> 2015 / 2016
Scaricata: 1617 -> 2016 / 2017
Scaricata: 1718 -> 2017 / 2018
Scaricata: 1819 -> 2018 / 2019
Scaricata: 1920 -> 2019 / 2020
Scaricata: 2021 -> 2020 / 2021
Scaricata: 2122 -> 2021 / 2022
Scaricata: 2223 -> 2022 / 2023
Scaricata: 2324 -> 2023 / 2024
Scaricata: 2425 -> 2024 / 2025


Stagioni 0304 e 0405 perche non me le riesce a parsare automanticamente il csv, le carico manualmente e le aggiungo al dizionario in un secondo momento

In [4]:
def read_fixed_csv(path: str) -> pd.DataFrame:
    '''
    Input: Path del CSV scaricato dal sito
    Output: df del campionato parsato correttamente 
    '''
    path = Path(path)
    fixed_path = path.with_name(path.stem + "_fixed.csv")

    with path.open("r", encoding="utf-8", errors="replace") as fin, \
         fixed_path.open("w", encoding="utf-8", newline="") as fout:
        for line in fin:
            # rimuove spazi + newline, poi toglie eventuali virgole finali
            cleaned = line.rstrip("\n\r").rstrip()
            cleaned = cleaned.rstrip(",")
            fout.write(cleaned + "\n")

    # ora le colonne saranno coerenti
    df = pd.read_csv(fixed_path)
    return df

df_0304 = read_fixed_csv(r"/Users/davide/Desktop/dav/statsXSerieA/I1(3).csv")
df_0405 = read_fixed_csv(r"/Users/davide/Desktop/dav/statsXSerieA/I1(4).csv")
print(df_0304.head())
print(df_0405.head())

  Div      Date  HomeTeam   AwayTeam  FTHG  FTAG FTR  HTHG  HTAG HTR  ...  \
0  I1  30/08/03   Reggina  Sampdoria     2     2   D     2     0   H  ...   
1  I1  31/08/03   Bologna      Parma     2     2   D     1     1   D  ...   
2  I1  31/08/03   Brescia     Chievo     1     1   D     0     1   A  ...   
3  I1  31/08/03     Inter     Modena     2     0   H     0     0   D  ...   
4  I1  31/08/03  Juventus     Empoli     5     1   H     1     0   H  ...   

   B365<2.5  GBAHH  GBAHA  GBAH  LBAHH  LBAHA  LBAH  B365AHH  B365AHA  B365AH  
0      1.83   2.00   1.80 -0.25   2.05   1.80 -0.25    2.075    1.825   -0.25  
1      1.80   2.10   1.75 -0.25   2.10   1.75 -0.25    1.750    2.150    0.00  
2      1.72   2.05   1.77 -0.25   2.10   1.75 -0.25    1.750    2.150    0.00  
3      1.90   1.85   1.95 -1.50   1.90   1.95 -1.50    1.925    1.975   -1.50  
4      2.00   1.85   1.95 -1.50   1.85   2.00 -1.50    1.875    2.025   -1.50  

[5 rows x 44 columns]
  Div      Date  HomeTeam  AwayTea

In [5]:
"""
Aggiorno il dizionario con i due dataframe dei due campionati mancanti

"""

campionati.update({"0304":df_0304})
campionati.update({"0405":df_0405})


In [6]:
def parseDateSafe(x):

    for fmt in ("%d/%m/%Y", "%d/%m/%y", "%Y-%m-%d", "%d-%m-%Y"):
        try:
            return datetime.strptime(str(x), fmt)
        except:
            continue
    return pd.NaT

def matchesBuilder(df: pd.DataFrame) -> pd.DataFrame:
    """
    In: dataframe
    Out: dataframe

    Pulisco il dataset che avevo in entrata andando a fare una feature selection di cio che serve a me
    """
   
    colonneDaMappare = {
        "HomeTeam": "HomeTeam",
        "AwayTeam": "AwayTeam",
        "FTHG": "FTHG",
        "FTAG": "FTAG",
        "Date": "Date",
    }

    out = df[list(colonneDaMappare.keys())].copy()

   # ho dovuto aggiungere la funzione parseDateSafe per evitare problemi tra formati 
    out["Date"] = out["Date"].apply(parseDateSafe)

    out = out.sort_values(["Date"], na_position="last").reset_index(drop=True)
    
    out["FTHG"] = pd.to_numeric(out["FTHG"], errors="coerce")
    out["FTAG"] = pd.to_numeric(out["FTAG"], errors="coerce")

    
    out = out.dropna(subset=["HomeTeam", "AwayTeam", "FTHG", "FTAG"]).reset_index(drop=True)

    return out

def classificaBuilder(df_season: pd.DataFrame):
    """
    In: dataframe
    Out: variabili winter,final che poi andro appaire in un secondo momento per creare il df finale 
    
    """
    m = matchesBuilder(df_season)

    teams = pd.unique(pd.concat([m["HomeTeam"], m["AwayTeam"]], ignore_index=True))
    n_teams = len(teams)
    half_games = n_teams - 1

    table = pd.DataFrame(index=teams, data={"GP": 0, "GF": 0, "GA": 0, "Pts": 0})
    winter = None

    for _, r in m.iterrows():
        h, a = r["HomeTeam"], r["AwayTeam"]
        hg, ag = int(r["FTHG"]), int(r["FTAG"])

        table.loc[[h, a], "GP"] += 1
        table.loc[h, "GF"] += hg
        table.loc[h, "GA"] += ag
        table.loc[a, "GF"] += ag
        table.loc[a, "GA"] += hg

        if hg > ag:
            table.loc[h, "Pts"] += 3
        elif hg < ag:
            table.loc[a, "Pts"] += 3
        else:
            table.loc[[h, a], "Pts"] += 1

        if winter is None and table["GP"].min() >= half_games:
            tmp = table.copy()
            tmp["GD"] = tmp["GF"] - tmp["GA"]
            winter = tmp.sort_values(["Pts", "GD", "GF"], ascending=False).index[0]

    tmp = table.copy()
    tmp["GD"] = tmp["GF"] - tmp["GA"]
    final = tmp.sort_values(["Pts", "GD", "GF"], ascending=False).index[0]

    return winter, final

In [7]:
results = []

for season, df in campionati.items():
    try:
        winter, final = classificaBuilder(df)
        results.append((season, winter, final, winter == final))
    except Exception as e:
        results.append((season, None, None, None))
        print("Errore su", season, ":", repr(e))

res = pd.DataFrame(results, columns=["season", "winter", "final", "same"]).sort_values("season")
print(res)

print("P(scudetto | campione d'inverno) =", res["same"].mean())

   season      winter     final   same
7    0001        Roma      Roma   True
8    0102        Roma  Juventus  False
9    0203       Milan  Juventus  False
30   0304        Roma     Milan  False
31   0405    Juventus  Juventus   True
10   0506    Juventus  Juventus   True
11   0607       Inter     Inter   True
12   0708       Inter     Inter   True
13   0809       Inter     Inter   True
14   0910       Inter     Inter   True
15   1011       Milan     Milan   True
16   1112    Juventus  Juventus   True
17   1213    Juventus  Juventus   True
18   1314    Juventus  Juventus   True
19   1415    Juventus  Juventus   True
20   1516      Napoli  Juventus  False
21   1617    Juventus  Juventus   True
22   1718      Napoli  Juventus  False
23   1819    Juventus  Juventus   True
24   1920    Juventus  Juventus   True
25   2021       Milan     Inter  False
26   2122       Inter     Milan  False
27   2223      Napoli    Napoli   True
28   2324       Inter     Inter   True
29   2425      Napoli    

In [8]:
processor = SerieADataProcessor(campionati)
setDati = processor.build_training_set()
setDati = setDati[['Team','Season','GapOnSecond','GD','WonScudetto']]
print(f"Dataset creato con {len(setDati)} record storici.")
print(setDati)

Dataset creato con 32 record storici.
          Team Season  GapOnSecond  GD  WonScudetto
0        Milan   9394            3  12            1
1     Juventus   9495            1  11            1
2        Milan   9596            1  14            1
3     Juventus   9697            4  11            1
4     Juventus   9798            1  25            1
5   Fiorentina   9899            3  13            0
6     Juventus   9900            1  14            0
7         Roma   0001            6  20            1
8         Roma   0102            4  17            0
9        Milan   0203            3  23            0
10    Juventus   0506           10  31            1
11       Inter   0607            9  25            1
12       Inter   0708            5  30            1
13       Inter   0809            3  18            1
14       Inter   0910            6  24            1
15       Milan   1011            4  17            1
16    Juventus   1112            1  20            1
17    Juventus   1213     

# ML: Classificazione

In [9]:
processor.features = ['GD', 'GapOnSecond']

X = setDati[processor.features]
y = setDati["WonScudetto"]
features = [c for c in setDati.columns if c not in ["Team", "Season", "WonScudetto"]]

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.80)

In [11]:
model = LogisticRegression()

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:,1]
preds = model.predict(X_test)

from sklearn.metrics import accuracy_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test, preds))
print("ROC AUC:", roc_auc_score(y_test, probs))

Accuracy: 0.8571428571428571
ROC AUC: 0.9


In [12]:
processor.set_trained_model(model)
inter2526 = {
    
    'GD': 26, 'GapOnSecond': 3
}
processor.predici_scudetto("Inter",inter2526)


--- Analisi Modello: Inter ---
Probabilità vittoria finale: 70.18%


np.float64(0.7018302432766732)

In [13]:
processor.features = ['GD', 'GapOnSecond']
X= setDati.drop(columns=["Team","Season","WonScudetto"])
y= setDati["WonScudetto"]

In [14]:
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

In [15]:
loo = LeaveOneOut()

model = LogisticRegression()

y_true = []
y_pred = []
y_prob = []

In [16]:
for train_index, test_index in loo.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)

    pred = model.predict(X_test)[0]
    prob = model.predict_proba(X_test)[0][1]

    y_pred.append(pred)
    y_prob.append(prob)
    y_true.append(y_test.values[0])

In [17]:
print("Accuracy:", accuracy_score(y_true, y_pred))
print("ROC AUC:", roc_auc_score(y_true, y_prob))

Accuracy: 0.625
ROC AUC: 0.5748792270531401


In [18]:
print("Probabilità media scudetto campione d'inverno:", np.mean(y_prob))

Probabilità media scudetto campione d'inverno: 0.718909365928592


In [19]:
model.fit(X, y)

inter = pd.DataFrame([{
    'GapOnSecond': 3,
    'GD': 26
}])

p=float(((model.predict_proba(inter)[0][1])).round(3)*100)
print(f"Probabilità che l'inter vinca lo scudetto: {p}%")

Probabilità che l'inter vinca lo scudetto: 69.3%
